In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow import keras 
from tensorflow.keras import layers 
from sklearn.model_selection import train_test_split


In [2]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings("ignore")
from keras.utils import image_dataset_from_directory

In [3]:
BATCH_SIZE = 32
IMG_SIZE = (256, 256)

In [4]:
train_ds = image_dataset_from_directory(
    './catdog/train',
    labels='inferred',
    label_mode='int', 
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True
)

Found 557 files belonging to 2 classes.


In [5]:
val_ds = image_dataset_from_directory(
    './catdog/test', # standard test split with labels if using a subset
    labels='inferred',
    label_mode='int',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False
)

Found 140 files belonging to 2 classes.


In [6]:
normalization_layer = keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [7]:
model = Sequential([
    # First Conv Block
    Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    MaxPooling2D(2, 2),
    
    # Second Conv Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Third Conv Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    # Flattening & Dense Layers
    Flatten(),
    Dense(128, activation='relu'),
    
    Dense(1, activation='sigmoid') # Binary output (0=Cat, 1=Dog)
])

In [8]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,839,105 (56.61 MB)

 Trainable params: 14,839,105 (56.61 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 85s 4s/step - accuracy: 0.5278 - loss: 1.3510 - val_accuracy: 0.5286 - val_loss: 0.6918
Epoch 2/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 74s 4s/step - accuracy: 0.5242 - loss: 0.6901 - val_accuracy: 0.5286 - val_loss: 0.6899
Epoch 3/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 78s 4s/step - accuracy: 0.6140 - loss: 0.6646 - val_accuracy: 0.5286 - val_loss: 0.6716
Epoch 4/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 73s 4s/step - accuracy: 0.6499 - loss: 0.5983 - val_accuracy: 0.6429 - val_loss: 0.6712
Epoch 5/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 86s 5s/step - accuracy: 0.7882 - loss: 0.4537 - val_accuracy: 0.6000 - val_loss: 0.7804
Epoch 6/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 132s 4s/step - accuracy: 0.8438 - loss: 0.3571 - val_accuracy: 0.6571 - val_loss: 0.8328
Epoch 7/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 89s 5s/step - accuracy: 0.9138 - loss: 0.2123 - val_accuracy: 0.6500 - val_loss: 1.2437
Epoch 8/10
18/18 ━━━━━━━━━━━━━━━━━━━━ 70s 4s/step - accuracy: 0.9408 - loss: 0.1239 - val_accuracy: 0.6286 - val_loss